# W01-B · Support-inbox challenge *(pairs, ~45 min)*

**Scenario.** *GharTech Appliances* (a fictional Indian e-tailer) wants to send its support tickets to an outside analytics vendor.
Legal says: **no personal data leaves the building.** Your pair has 45 minutes to build a regex pipeline that:

1. **extracts** order IDs, emails, phone numbers, rupee amounts and PIN codes into structured columns
2. **redacts** personal data before the tickets are shared
3. *(stretch)* **routes** each ticket to the right team using keywords

Every message has **gold labels**, so you can score yourself. At the share-out, each pair reports its **F1 per task**.

> All 300 messages are synthetic: every name, email and phone number was randomly generated.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 160)
DATA = "../data/"   # notebooks live in ADS6703/w01/, data in ADS6703/data/

from collections import Counter

# keep_default_na=False: empty cells stay "" rather than NaN, so every gold column can be split on "|"
# dtype=str: keep every column as text, exactly as written
inbox = pd.read_csv(DATA + "support_inbox.csv", keep_default_na=False, dtype=str)
print(inbox.shape)
inbox[["message_id", "text"]].head(5)

### The scorer (read, don't edit)

For each message we compare **your list** of extracted items with the **gold list**:
- **true positive (TP)**: you found it and it's in gold
- **false positive (FP)**: you found it but it's not in gold (over-matching)
- **false negative (FN)**: it's in gold but you missed it (under-matching)

$$\text{precision} = \frac{TP}{TP+FP} \qquad \text{recall} = \frac{TP}{TP+FN} \qquad F_1 = \frac{2PR}{P+R}$$

In [ ]:
def score(pred_lists, gold_col, show_errors=0):
    # pred_lists: one list of extracted items per message; gold_col: correct items joined with "|"
    tp = fp = fn = 0
    errors = []
    for i, (pred, gold) in enumerate(zip(pred_lists, inbox[gold_col])):
        # Counters rather than sets, so an item that appears twice has to be found twice
        g = Counter(gold.split("|")) if gold else Counter()
        p = Counter(pred)
        # p & g: found and correct (TP) · p - g: found but wrong (FP) · g - p: missed (FN)
        tp += sum((p & g).values()); fp += sum((p - g).values()); fn += sum((g - p).values())
        if p != g:
            errors.append((inbox.at[i, "message_id"], sorted(p - g), sorted(g - p)))
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    print(f"{gold_col:15s}  precision {P:.3f}   recall {R:.3f}   F1 {F:.3f}")
    for mid, extra, missed in errors[:show_errors]:
        print(f"   {mid}: extra {extra}  missed {missed}")
    return round(F, 3)

results = {}

## Task 1 · Order IDs (warm-up, 5 min)

Order IDs look like `GT-2026-0048213`. Watch out for old complaint numbers.

In [ ]:
pattern = r"GT-\d+-\d+"      # a first attempt: run it and read the errors
results["orders"] = score(inbox["text"].str.findall(pattern), "gold_order_ids", show_errors=3)

In [ ]:
order_pattern = r"..."   # ✍️ improve it
results["orders"] = score(inbox["text"].str.findall(order_pattern), "gold_order_ids", show_errors=3)

## Task 2 · Emails (8 min)

Hint: run the obvious pattern `[\w.]+@[\w.]+` first and **read the errors**. What happens when an email ends a sentence?

In [ ]:
email_pattern = r"[\w.]+@[\w.]+"   # ✍️ start here, then fix it
results["emails"] = score(inbox["text"].str.findall(email_pattern), "gold_emails", show_errors=3)

## Task 3 · Indian mobile numbers (12 min, the hard one)

Customers write the same number many ways:

`9822967803` · `+91 9822967803` · `+91-98229-67803` · `09822967803` · `98229 67803` · `+919822967803` · `982 296 7803`

Gold stores the **10-digit** number only. Things to know:
- Indian mobile numbers start with **6, 7, 8 or 9**
- ticket numbers, serial numbers (`SN…`) and PIN codes are **not** phone numbers

We've written the normaliser for you. Your job is the pattern.

In [ ]:
def normalise_phone(match):
    # "+91-98229-67803" -> "919822967803" -> last 10 digits "9822967803"
    return re.sub(r"\D", "", match)[-10:]      # keep the last 10 digits

naive = inbox["text"].apply(lambda s: [normalise_phone(m) for m in re.findall(r"\d{10}", s)])
results["phones"] = score(naive, "gold_phones", show_errors=3)

In [ ]:
# Hint: lookarounds (?<!...) and (?!...) check what comes before or after a match without consuming it
phone_pattern = r"..."   # ✍️ your pattern (use (?: ) groups so findall returns the whole match)
preds = inbox["text"].apply(lambda s: [normalise_phone(m) for m in re.findall(phone_pattern, s)])
results["phones"] = score(preds, "gold_phones", show_errors=5)

## Task 4 · Rupee amounts (8 min)

Formats seen: `₹12,499` · `₹ 12,499` · `Rs. 12,499` · `Rs 12499` · `INR 12,499` · `Rs.12499/-` · `₹1680.99`

Gold stores amounts as plain numbers with two decimals: `12499.00`, `1680.99`.
Complete the pattern **and** the `to_number` function.

In [ ]:
amount_pattern = r"..."   # ✍️ hint: two capturing groups, (rupees)(optional .paise)

# ("12,499", "") -> "12499.00" · ("1680", ".99") -> "1680.99"
def to_number(rupees, paise):
    ...   # ✍️ return a string like "12499.00"

preds = inbox["text"].apply(lambda s: [to_number(r, p) for r, p in re.findall(amount_pattern, s)])
results["amounts"] = score(preds, "gold_amounts", show_errors=5)

## Task 5 · PIN codes (3 min)

Indian PIN codes are six digits and never start with 0.

In [ ]:
pin_pattern = r"..."   # ✍️
results["pincodes"] = score(inbox["text"].str.findall(pin_pattern), "gold_pincodes", show_errors=3)

## Task 6 · Redact before sharing (6 min)

Write `redact(text)` so that emails become `[EMAIL]`, phones become `[PHONE]` and PIN codes become `[PIN]`.
Also remove the customer's name in the sign-off (`Regards, Firstname Lastname`) and replace it with `[NAME]`.

Keep order IDs: the vendor needs them to link tickets.

In [ ]:
def redact(text):
    ...   # ✍️ chain re.sub calls using your patterns from above
    return text

inbox["redacted"] = inbox["text"].apply(redact)
inbox[["text", "redacted"]].head(3)

In [ ]:
# ✅ leak check: does any gold email or phone survive redaction?
leaks = 0
for _, row in inbox.iterrows():
    digits = re.sub(r"\D", "", row["redacted"])        # digits only, so a spaced-out "98229 67803" still counts
    leaks += any(e and e in row["redacted"] for e in row["gold_emails"].split("|"))
    leaks += any(p and p in digits for p in row["gold_phones"].split("|"))
print(f"messages leaking an email or phone: {leaks} / {len(inbox)}")
print("✅ clean!" if leaks == 0 else "❌ keep going: print a few leaking rows to see why")

> 💬 **For the share-out:** Look at `"Company name: Khan Traders"` in some messages. Did your redactor catch that surname?
> Could **any** regex reliably find all names? What would you need instead?

## Task 7 · Stretch: rule-based ticket routing

Route each ticket to `refund`, `installation`, `delivery`, `warranty` or `other` using keyword regexes.
Check your accuracy against `inbox["intent"]`.

In [ ]:
rules = {
    "refund": r"...",          # ✍️
    "installation": r"...",
    "delivery": r"...",
    "warranty": r"...",
}

def route(text):
    for intent, pattern in rules.items():
        if re.search(pattern, text, flags=re.I):
            return intent
    return "other"

inbox["routed"] = inbox["text"].apply(route)
print("accuracy:", (inbox["routed"] == inbox["intent"]).mean().round(3))
pd.crosstab(inbox["intent"], inbox["routed"])   # rows: true intent · columns: your routing

> 💬 Try reordering the rules. Why does the order change your accuracy? With 50 intents and real customer language,
> would you still maintain this by hand? *(Workshop 4 shows how to **learn** these rules from data instead.)*

## Your scoreboard, for the share-out

In [ ]:
pd.Series(results, name="F1").to_frame()

## Ethics check: before you ship this 🛑

India's **Digital Personal Data Protection Act, 2023** treats names, phone numbers and emails as personal data.
Talk these through with your partner. We will discuss them at the share-out:

1. Your redactor scored F1 = 0.98 on phones. On 1 million tickets a year, how many phone numbers leak? Is that acceptable?
2. Who should decide that threshold: you, legal, or the customer?
3. The vendor wants to send the redacted tickets to an **LLM API** for summarisation. What else might still identify a customer?